# 01 — QFT Circuit Optimization with WestQuant Open

This notebook demonstrates how to use WestQuant Open to:
1. Build a Quantum Fourier Transform (QFT) circuit
2. Analyze its metrics (depth, 2-qubit gates, size)
3. Generate multiple compiled representations
4. Compare optimization levels and find the best representation

**Key idea:** The same quantum algorithm can have many valid circuit representations. WestQuant searches this space systematically.

In [ ]:
# Install WestQuant Open (uncomment if needed)
# !pip install westquant[qiskit]

## Step 1: Build a QFT Circuit

In [ ]:
from qiskit import QuantumCircuit
from qiskit.circuit.library import QFTGate
from westquant_qiskit import circuit_metrics

# Build a 6-qubit QFT circuit
n_qubits = 6
qc = QuantumCircuit(n_qubits)
qc.append(QFTGate(n_qubits), range(n_qubits))
qc = qc.decompose(reps=3)

# Analyze baseline metrics
metrics = circuit_metrics(qc)
print("=== Baseline QFT Circuit ===")
print(f"  Qubits:        {metrics['n_qubits']}")
print(f"  Depth:         {metrics['depth']}")
print(f"  2-qubit gates: {metrics['two_qubit_gates']}")
print(f"  Total gates:   {metrics['size']}")
print(f"  SWAP gates:    {metrics['swap_gates']}")
print(f"  Operations:    {metrics['operations']}")
qc.draw(output='mpl', fold=120)

## Step 2: Import into WestQuant's Framework-Neutral Representation

In [ ]:
from westquant_qiskit import QiskitAdapter

# Import the Qiskit circuit into WestQuant's framework-neutral WQIR
adapter = QiskitAdapter()
representation = adapter.import_native(qc)

print("=== WestQuant Representation ===")
print(f"  ID:           {representation.id}")
print(f"  Kind:         {representation.kind.name}")
print(f"  Framework:    {representation.framework}")
print(f"  Payload keys: {list(representation.payload.keys())}")
print(f"  Instructions: {len(representation.payload.get('instructions', []))}")
print(f"  Metrics:      {representation.payload['metrics']}")

## Step 3: Generate Multiple Compiled Representations

WestQuant's `generate_training_data` varies over optimization levels, basis gates, and routing methods to produce multiple representations of the same algorithm. Each representation is a different point in the compilation space.

In [ ]:
from westquant import generate_training_data

# Generate multiple representations by varying compilation config
samples = generate_training_data(
    qc,
    framework="qiskit",
    samples=50,
    seed=42,
    optimization_levels=[0, 1, 2, 3],
    basis_gates_options=[
        ["cx", "u3", "u1", "u2"],
        ["cx", "rz", "sx", "x"],
        ["ecr", "rz", "sx", "x"],
    ],
)

print(f"Generated {len(samples)} representations")
print()
for i, s in enumerate(samples[:5]):
    print(f"  [{i}] {s.representation}: depth={s.depth}, 2q={s.two_qubit_gates}, size={s.size}")

## Step 4: Find the Best Representation

In [ ]:
import pandas as pd

# Convert to DataFrame for analysis
data = []
for s in samples:
    data.append({
        "representation": s.representation,
        "depth": s.depth,
        "two_qubit_gates": s.two_qubit_gates,
        "size": s.size,
        "swap_gates": s.swap_gates,
        "num_qubits": s.num_qubits,
    })
df = pd.DataFrame(data)

# Find the best representation (lowest 2-qubit gate count)
best = df.loc[df["two_qubit_gates"].idxmin()]
worst = df.loc[df["two_qubit_gates"].idxmax()]

print("=== Best Representation (fewest 2Q gates) ===")
print(f"  {best['representation']}")
print(f"  Depth:    {best['depth']}")
print(f"  2Q gates: {best['two_qubit_gates']}")
print(f"  Size:     {best['size']}")
print()
print("=== Worst Representation (most 2Q gates) ===")
print(f"  {worst['representation']}")
print(f"  Depth:    {worst['depth']}")
print(f"  2Q gates: {worst['two_qubit_gates']}")
print(f"  Size:     {worst['size']}")
print()
improvement = (worst['two_qubit_gates'] - best['two_qubit_gates']) / worst['two_qubit_gates'] * 100
print(f"2Q gate reduction: {improvement:.1f}%")
df.sort_values("two_qubit_gates").head(10)

In [ ]:
import matplotlib.pyplot as plt

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

# Depth distribution
ax1.hist(df["depth"], bins=20, edgecolor="black", color="steelblue")
ax1.axvline(best["depth"], color="green", linestyle="--", label=f"Best: {best['depth']}")
ax1.axvline(worst["depth"], color="red", linestyle="--", label=f"Worst: {worst['depth']}")
ax1.set_xlabel("Circuit Depth")
ax1.set_ylabel("Count")
ax1.set_title("QFT-6 Depth Distribution Across Representations")
ax1.legend()

# 2Q gate distribution
ax2.hist(df["two_qubit_gates"], bins=20, edgecolor="black", color="coral")
ax2.axvline(best["two_qubit_gates"], color="green", linestyle="--", label=f"Best: {best['two_qubit_gates']}")
ax2.axvline(worst["two_qubit_gates"], color="red", linestyle="--", label=f"Worst: {worst['two_qubit_gates']}")
ax2.set_xlabel("2-Qubit Gates")
ax2.set_ylabel("Count")
ax2.set_title("QFT-6 2Q Gate Distribution Across Representations")
ax2.legend()

plt.tight_layout()
plt.savefig("qft_optimization.png", dpi=150, bbox_inches="tight")
plt.show()
print("Saved: qft_optimization.png")

## Summary

We took a single QFT circuit and generated **multiple valid representations** by varying the compilation configuration. The best representation uses **fewer 2-qubit gates** — directly translating to lower error on real hardware.

This is the core idea of WestQuant Open: **search the representation, not just the parameters.**

### Next steps
- **02_generate_ml_dataset.ipynb** — Scale this up to generate thousands of ML training examples
- **04_representation_search.ipynb** — Systematic search over transformation sequences